# Data Loading

In [2]:
import sys
sys.path.insert(0, "D:\\search-mlops\\wurth-search")

from services.ingestion.pipeline import preprocess_record

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import pandas as pd
import numpy as np

d:\search-mlops\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True
)
print(f"Loaded {len(dataset)} records")

2026-03-29 19:19:45,507  INFO      httpx  HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023 "HTTP/1.1 200 OK"
2026-03-29 19:19:49,318  INFO      httpx  HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/revision/main "HTTP/1.1 200 OK"
2026-03-29 19:19:49,503  INFO      httpx  HTTP Request: GET https://huggingface.co/api/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main?recursive=false&expand=false "HTTP/1.1 200 OK"


Loaded 112590 records


In [4]:
# Step 2 — Convert directly to DataFrame ✅ (no cache_files needed)
df = dataset.to_pandas()
print(f"DataFrame shape: {df.shape}")
print(df.columns.tolist())

DataFrame shape: (112590, 16)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [5]:
records = [r for r in df.to_dict(orient="records")]

# Example:

# If df is:

#    name   age
# 0  Alice   25
# 1  Bob     30

# Then:

# df.to_dict(orient="records")

# gives:

# [
#     {"name": "Alice", "age": 25},
#     {"name": "Bob", "age": 30}
# ]

In [6]:
docs = []
skipped = 0
for raw in records:
    doc = preprocess_record(raw)
    if doc is None:
        skipped += 1
        continue
    docs.append(doc)

In [7]:
print(f"Processed: {len(docs)}, Skipped: {skipped}")


Processed: 112590, Skipped: 0


In [8]:
df_clean = pd.DataFrame(docs)
df_clean.to_parquet("processed_products.parquet", index=False)

print(f"Saved {len(df_clean)} records")

Saved 112590 records


In [9]:
df_clean = pd.read_parquet("processed_products.parquet")

# Null check
print("Null values per column:")
print(df_clean.isnull().sum())

print("\n")

print("Embedding text length stats:")
print(df_clean['embedding_text'].str.len().describe())
empty_emb = df_clean[df_clean['embedding_text'].str.strip() == ""]
print(f"Empty embedding_text: {len(empty_emb)}")

print("\n")

# Price sanity
print("Price stats:")
print(df_clean['price'].describe())

print("\n")

# Rating sanity
print("Average rating stats:")
print(df_clean['average_rating'].describe())

print("\n")

# Category distribution
print("Main category distribution:")
print(df_clean['main_category'].value_counts())

Null values per column:
_id                      0
parent_asin              0
title                    0
description              0
features                 0
main_category            0
sub_category             0
store                    0
price                94886
average_rating           0
rating_number            0
primary_image_url        0
embedding_text           0
dtype: int64


Embedding text length stats:
count    112590.000000
mean        260.071516
std         413.785957
min           0.000000
25%          83.000000
50%         139.000000
75%         188.000000
max       10209.000000
Name: embedding_text, dtype: float64
Empty embedding_text: 9


Price stats:
count    17704.00000
mean        27.25573
std         50.47202
min          0.01000
25%          9.99000
50%         16.99000
75%         29.90000
max       2548.98000
Name: price, dtype: float64


Average rating stats:
count    112590.000000
mean          3.883488
std           0.874384
min           1.000000
25%      

##### **9 Empty embedding_text Records** 
1. These will produce meaningless zero vectors when embedded bad for search quality
2. Its better to remove these 9 products

In [10]:
# Identify the empty embedding_text records
empty = df_clean[df_clean['embedding_text'].str.strip() == ""]
print(empty[['parent_asin', 'title', 'description', 'features']])

# Drop the empty embedding_text records
df_clean = df_clean[df_clean['embedding_text'].str.strip() != ""]
print(f"After removing empty embedding_text: {len(df_clean)} records")


      parent_asin title description features
14147  B01K1GXQ6M                           
14148  B08C2GDFYG                           
18360  B0146F7B92                           
18395  B00SKB35R6                           
18397  B07C851DK1                           
18402  B082HBNFLN                           
19345  B004PLP7SK                           
19346  B089ZNHRTT                           
30846  B07S3QJWHQ                           
After removing empty embedding_text: 112581 records


##### **94,886 Missing Prices (~84% null!)**
1. This is expected for Amazon data, since many listings don't publish prices
2. We have decided to leave as None since it will be handled by OpenSearch mapping as nullable float
3. Price filters in the API will simply exclude these docs.

4. For example when users asks like I want T-shirts below 50 Euros. These price filter will be applied only on the 16% dataset. Once the filter on price is released, the filter will be applied on the entire dataset.

In [11]:
print(df_clean['main_category'].unique())
# Confirm these are the only two values


<ArrowStringArray>
['All Beauty', 'Premium Beauty']
Length: 2, dtype: str


Svae the Final Cleaned Dataset

In [12]:
df_clean = df_clean[df_clean['embedding_text'].str.strip() != ""].reset_index(drop=True)
df_clean.to_parquet("processed_products.parquet", index=False)
print(f"Final clean records: {len(df_clean)}")  # expect 112,581


Final clean records: 112581


#### Generate Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
texts = df_clean['embedding_text'].tolist()

model_100_texts = texts[:100]

print(f"Encoding {len(model_100_texts)} documents...")



In [ ]:
vectors = model.encode(
    model_100_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings separately
np.save("product_embeddings.npy", vectors)
print(f"Saved embeddings shape: {vectors.shape}")  # should be (112590, 384)


#####  Save a Sample for API Testing

In [ ]:
sample_size = 100
sample_docs = docs[:sample_size]
sample_vectors = vectors[:sample_size]

import json
with open("sample_products.jsonl", "w") as f:
    for doc, vec in zip(sample_docs, sample_vectors.tolist()):
        doc['embedding_vector'] = vec
        f.write(json.dumps(doc) + "\n")

print(f"Saved {sample_size} sample documents with vectors")


##### Verify One Doc is Fully OpenSearch-Ready

In [ ]:
sample = docs[0].copy()
sample['embedding_vector'] = vectors[0].tolist()

# All values must be JSON serialisable
print(json.dumps(sample, indent=2))

# Vector dimension must be 384
assert len(sample['embedding_vector']) == 384
print("✅ Document is OpenSearch-ready")


In [1]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import json

# Load sample
with open("sample_products.jsonl") as f:
    sample_docs = [json.loads(line) for line in f]

doc_vectors = np.array([doc["embedding_vector"] for doc in sample_docs])
titles = [doc["title"] for doc in sample_docs]

# Query
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
query = input("Enter search query: ")
q_vector = model.encode(query, normalize_embeddings=True).reshape(1, -1)

# Cosine similarity
scores = cosine_similarity(q_vector, doc_vectors)[0]
top5 = np.argsort(scores)[::-1][:5]

print(f"Query: '{query}'\n")
for i in top5:
    print(f"Score: {scores[i]:.4f} | {titles[i][:170]}")

d:\search-mlops\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 869.47it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: 'skin creame'

Score: 0.7646 | Sharonelle Natural Cream Soft Wax for Sensitive Skin in 14 oz. - 4 cans
Score: 0.7428 | reskin TS Propolis Serum Even Skin-tone Moisture Care
Score: 0.7100 | Clear Essence Specialist Skin Care Body Oil, 8 Ounce - Soothing Body Moisturizer and Toner
Score: 0.7063 | Zoella Beauty Tutti Fruity Foam Sweet Foam Shower Gel 250ml
Score: 0.7018 | FeiDing Liquid Highlighter Illuminator Rose Gold Body Shimmer Oil Spray For Face Arm Leg Shimming Glow Lotion For Dark Skin Glitter For Body Shimmer (Rose Gold)
